<div class="align-center"> <a href="https://github.com/unslothai/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a> <a href="https://ollama.com/"><img src="https://raw.githubusercontent.com/unslothai/unsloth/main/images/ollama.png" height="44"></a>

Чтобы установить Unsloth на свой компьютер, следуйте инструкциям по установке на нашей странице Github [здесь](https://github.com/unslothai/unsloth#installation-instructions---conda).

Вы научитесь подготавливать данные, как дообучать модель, как запускать модель и как экспортировать в Ollama и Hugging Face!

Ускорение дообучения моделей!

С помощью Unsloth вы можете автоматически дообучать модель и создавать Modelfile, для экспорта в Ollama

[НОВОЕ] Теперь поддерживается загрузка CSV и Excel файлов — можете попробовать на датасете Titanic.

In [ ]:
%%capture
!pip install unsloth
# Последняя сборка
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

* Поддержка Llama, Mistral, Phi-3, Gemma, Yi, DeepSeek, Qwen, TinyLlama, Vicuna, Open Hermes и другие модели.
* Поддержка 16-битный LoRA и 4-битный QLoRA. Оба варианта в 2 раза быстрее.
* Параметр max_seq_length можно установить любой, поскольку используется автоматическое масштабирование RoPE с помощью метода [kaiokendev's](https://kaiokendev.github.io/til)

* [НОВОЕ] Ускорена работа Phi-3 Medium / Mini в 2 раза!

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Выберите любое значение! поддержка масштабирования RoPE.
dtype = None  # None для автоопределения. Float16 для Tesla T4, V100, Bfloat16 для Ampere+.
load_in_4bit = True  # Использовать 4-битное квантование для уменьшения использования памяти. Может быть False.

# список доступных моделей
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",      # Новая Mistral v3, в 2 раза быстрее!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",           # Llama-3, в 2 раза быстрее!
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",        # Phi-3, в 2 раза быстрее!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",             # Gemma, в 2.2 раза быстрее!
]  # Больше моделей можно найти на https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # Используйте токен, если работаете с моделями, которые требуют gated access (например, meta-llama/Llama-2-7b-hf)
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Добавляем LoRA, чтобы обновлять часть весов

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # Выберите любое число > 0! Рекомендуемые значения: 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,  # Поддерживает любые значения, но 0 оптимально
    bias = "none",     # Поддерживает любые значения, но "none" оптимально
    # [НОВОЕ] "unsloth" использует на 30% меньше VRAM, позволяет использовать в 2 раза большие размеры батчей!
    use_gradient_checkpointing = "unsloth",  # True или "unsloth" для очень длинного контекста
    random_state = 3407,
    # use_rslora = False,  # Поддержка rank stabilized LoRA
    # loftq_config = None,  # И LoftQ
)


Unsloth 2025.2.15 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


### Подготовка данных

Используем набор данных с HF, он уже представлен в необходимом виде и явлется объектом класса datasets

[Набор данных](https://huggingface.co/datasets/Amod/mental_health_counseling_conversations?row=1) представляет собой коллекцию вопросов и ответов, взятых с двух платформ для онлайн-консультирования и терапии. Вопросы охватывают широкий спектр тем, связанных с психическим здоровьем, а ответы даны квалифицированными психологами.

Набор данных предназначен для тонкой настройки языковых моделей с целью улучшения их способности предоставлять консультации по вопросам психического здоровья.

Это может пригодиться при создании чат-ботов, оказывающих консультации людям


In [ ]:
from datasets import load_dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split = "train")
print(dataset.column_names)

README.md:   0%|          | 0.00/2.82k [00:00<?, ?B/s]

combined_dataset.json:   0%|          | 0.00/4.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

['Context', 'Response']


Также разделим датасет на две части - обучающую и тестовую выборки

In [ ]:
# Разделяем датасет на train и test
train_test_split = dataset.train_test_split(test_size=0.2)  # 20% данных на тестовый набор

train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))

Train dataset size: 2809
Test dataset size: 703


Может возникунть проблема, которая заключается в том, что некоторые наборы данных содержит несколько столбцов. Для того чтобы Ollama и llama.cpp работали как чат-бот ChatGPT, необходимо, чтобы в наборе данных было только два столбца — instruction (инструкция) и output (выход).

Для решения этой задачи мы сделаем следующее:

* Объединим все столбцы в один prompt.
* Используем функцию *to_sharegpt*, чтобы выполнить процесс слияния столбцов!

Чтобы объединить несколько столбцов в один, используйте **merged_prompt**.

* Заключите все столбцы в фигурные скобки {}.
* Необязательный текст должен быть заключен в [[]]. Например, если столбец "Pclass" пуст, функция слияния не покажет этот текст и пропустит его. Это полезно для наборов данных с отсутствующими значениями.
* Вы можете выбрать все столбцы или несколько!
* Выберите столбец с предсказаниями в output_column_name. Для нашего набора данных это - Response
* *Дополнительно*: Чтобы дообучение поддерживало несколько обменов сообщениями (как в ChatGPT), нужно создать "фейковый" набор данных с несколькими сообщениями в диалоге — для этого мы используем conversation_extension, чтобы случайным образом выбрать несколько разговоров из набора данных и объединить их в один разговор

In [ ]:
from unsloth import to_sharegpt
dataset = to_sharegpt(
    dataset,
    merged_prompt = "{Context}",
    output_column_name = "Response",
    # conversation_extension = 1,
)

Merging columns:   0%|          | 0/3512 [00:00<?, ? examples/s]

Converting to ShareGPT:   0%|          | 0/3512 [00:00<?, ? examples/s]

Используйте **standardize_sharegpt**, чтобы привести набор данных к нужному виду!

In [ ]:
from unsloth import standardize_sharegpt
dataset = standardize_sharegpt(dataset)

Standardizing format:   0%|          | 0/3512 [00:00<?, ? examples/s]

### Настройка шаблона (промпта) чата

In [ ]:
chat_template = """Below are some conversations. Write responses that appropriately complete each request.

### Question:
{INPUT}

### Corresponding answer:
{OUTPUT}"""

from unsloth import apply_chat_template
dataset = apply_chat_template(
    dataset,
    tokenizer = tokenizer,
    chat_template = chat_template,
    # default_system_message = "You are a helpful assistant", << [OPTIONAL]
)

Unsloth: We automatically added an EOS token to stop endless generations.


Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['conversations', 'text'],
    num_rows: 3512
})

Посмотрим, как модель отвечает на вопросы до дообучения

In [ ]:
import time

FastLanguageModel.for_inference(model) # Ускорение модели
messages = [
    {"role": "user", "content": "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?"},
]
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# Время начала генерации
start_time = time.time()
# Генерируем текст
output_ids = model.generate(input_ids, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id)

# Время окончания генерации
end_time = time.time()

# Время, затраченное на генерацию
generation_time = end_time - start_time
print(f"Время генерации: {generation_time:.2f} секунд")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Время генерации: 24.03 секунд


In [ ]:
# Декодируем токены в строку
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output_text)

Below are some conversations. Write responses that appropriately complete each request.

### Question:
I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?

### Corresponding answer:
I'm sorry to hear that you're going through this. I know how it feels to be in a dark place and not know how to get out of it. It's okay to feel this way. You're not alone. There are people who care about you and want you to feel better. I know it may not feel like it now, but you are worth it. You are worth it. You are worth it. You are worth it. You are worth it. You are worth it


In [ ]:
# Декодируем токены в строку
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output_text)

Below are some conversations. Write responses that appropriately complete each request.

### Question:
I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?

### Corresponding answer:
I'm sorry to hear that you're going through this. I know how it feels to be in a dark place and not know how to get out of it. It's okay to feel this way. You're not alone. There are people who care about you and want you to feel better. I know it may not feel like it now, but you are worth it. You are worth it. You are worth it. You are worth it. You are worth it. You are worth it


In [ ]:
print(output_ids)

tensor([[128000,  39314,    527,   1063,  21633,     13,   9842,  14847,    430,
          36001,   4686,   1855,   1715,    382,  14711,  16225,    512,     40,
           2846,   2133,   1555,   1063,   2574,    449,    856,  16024,    323,
           7182,     13,    358,  20025,   6212,    323,    358,    656,   4400,
            719,   1781,    922,   1268,    358,   2846,  66121,    323,   1268,
            358,  13434,    956,    387,   1618,     13,    358,   3077,   2646,
           6818,    477,  93439,  18639,     13,    358,   3077,   2744,   4934,
            311,   5155,    856,   4819,     11,    719,    358,   2646,    636,
           2212,    311,    433,     13,   2650,    649,    358,   2349,    856,
           8430,    315,   1694,  66121,    311,   5127,   1980,  14711,  52042,
            287,   4320,    512,     40,   2846,  14931,    311,   6865,    430,
            499,   2351,   2133,   1555,    420,     13,    358,   1440,   1268,
            433,  11321,    

In [ ]:
print(dataset['text'][1])  # Покажет первые 10 элементов из столбца 'text'

<|begin_of_text|>Below are some conversations. Write responses that appropriately complete each request.

### Question:
("I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",)

### Corresponding answer:
Hello, and thank you for your question and seeking advice on this. Feelings of worthlessness is unfortunately common. In fact, most people, if not all, have felt this to some degree at some point in their life. You are not alone. Changing our feelings is like changing our thoughts - it's hard to do. Our minds are so amazing that the minute you change your thought another one can be right there to take it's place. Without your permission, another thought can just pop in there. The new thought may feel worse than 

### Дообучение модели



Теперь используем **SFTTrainer** из библиотеки Huggingface TRL! Подробнее в документации: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer).

Мы проведем всего 60 шагов во время дообучения, но вы можете установить num_train_epochs=1 для полноценного обучения и отключить max_steps=None.

Также есть поддержка DPOTrainer от TRL!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    # compute_metrics=compute_metrics,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        max_steps = 60,
        # num_train_epochs = 3, # Для длительного дообучения
        learning_rate = 0.001,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Converting train dataset to ChatML (num_proc=2):   0%|          | 0/3512 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=2):   0%|          | 0/3512 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/3512 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/3512 [00:00<?, ? examples/s]

In [ ]:
#@title Состояние системы
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
5.83 GB of memory reserved.


In [ ]:
import wandb
# Отключение логирования wandb
wandb.init(mode='disabled')

Запуск дообучения

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 3,512 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 60
 "-____-"     Number of trainable parameters = 41,943,040
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss
1,2.788700
2,2.430100
3,2.507000
4,2.404000
5,2.216400
6,2.202800
7,2.095100
8,2.207600
9,2.256000
10,2.182200


In [ ]:
#@title Показать итоговую статистику по расходу памяти и времени
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

647.3574 seconds used for training.
10.79 minutes used for training.
Peak reserved memory = 7.328 GB.
Peak reserved memory for training = 1.498 GB.
Peak reserved memory % of max memory = 49.712 %.
Peak reserved memory for training % of max memory = 10.162 %.


### Инференс модели

Посмотрим, как модель генерирует текст после дообучения

In [ ]:
import time

FastLanguageModel.for_inference(model)
messages = [
    {"role": "user", "content": "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?"},
]
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# Время начала генерации
start_time = time.time()
# Генерируем текст
output_ids = model.generate(input_ids, max_new_tokens=150, pad_token_id=tokenizer.eos_token_id)

# Время окончания генерации
end_time = time.time()

# Время, затраченное на генерацию
generation_time = end_time - start_time
print(f"Время генерации: {generation_time:.2f} секунд")

Время генерации: 10.41 секунд


In [ ]:
# Декодируем токены в строку
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output_text)

Below are some conversations. Write responses that appropriately complete each request.

### Question:
I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?

### Corresponding answer:
I'm sorry to hear about your struggles. I think it is great that you are reaching out and want to change your feelings. I believe that everyone has value, but I also believe that we have to create our own value. There are many things that can be done to help you feel better about yourself. I would recommend talking to a therapist about your feelings. You may also want to try out some self-help books or apps. I would also recommend that you try to get more sleep. I know it is hard to get enough sleep when you are feeling this way, but it is i

### Сохранение и загрузка дообученных моделей


#### Сохранение только LoRA весовых коэффициентов

Чтобы сохранить итоговую модель в виде LoRA адаптеров, используйте push_to_hub от Huggingface для сохранения на HF или save_pretrained для локального сохранения.

[ПРИМЕЧАНИЕ] Это сохраняет только LoRA адаптеры, а не всю модель.
Чтобы сохранить модель в формате 16bit или GGUF, прокрутите вниз!

In [ ]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')


Теперь, если вы хотите загрузить адаптеры LoRA, которые мы только что сохранили для выводов,  `False` замените на `True`:




In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # ваша дообученная модель
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)
pass

messages = [
    {"role": "user", "content": "Describe anything special about a sequence. Your input is 1, 1, 2, 3, 5, 8,"},
]
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids, streamer = text_streamer, max_new_tokens = 128, pad_token_id = tokenizer.eos_token_id)

It depends on what you want to know.  If you want to know what the next number in the sequence is, you can use the Fibonacci formula.  If you want to know how many numbers are in the sequence, you can use the Fibonacci formula and keep adding numbers until the sum of the two previous numbers equals the current number.  If you want to know how many numbers are in the sequence, you can use the Fibonacci formula and keep adding numbers until the sum of the two previous numbers equals the current number.<|end_of_text|>


<a name="Ollama"></a>
#### Ollama Support

Unsloth позволяет дообучать модели, создавать Modelfile и экспортировать их в Ollama! Это значительно упрощает процесс дообучения и обеспечивает плавный рабочий процесс от Unsloth к Ollama!

Давайте сначала установим Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
############################################################################################# 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Теперь сохраним модель в формате GGUF

Используйте метод save_pretrained_gguf для локального сохранения модели и push_to_hub_gguf для загрузки на Hugging Face Hub.
По умолчанию модель сохраняется в формате q8_0.

Некоторые поддерживаемые методы квантования, больше - [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options):

q8_0 — Быстрая конвертация. Требует много ресурсов, но в целом приемлемо.

q4_k_m — Рекомендуемый метод. Использует Q6_K для половины тензоров attention.wv и feed_forward.w2, остальное — Q4_K.

q5_k_m — Рекомендуемый метод. Использует Q6_K для половины тензоров attention.wv и feed_forward.w2, остальное — Q5_K.

In [ ]:
# Save to 8bit Q8_0
if True: model.save_pretrained_gguf("model", tokenizer,)
# За HF токеном идем -  https://huggingface.co/settings/tokens
# Измени hf на своем имя!
# Создай в своем профиле модель с именем model , чтобы сохранить в нее модель
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Сохранить в 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Сохранить в q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Сохранить в различные форматы GGUF
if False:
    model.push_to_hub_gguf(
        "hf/model", # Измени hf на своем имя!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # За HF токеном идем -  https://huggingface.co/settings/tokens
    )

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.
Unsloth: You have 1 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which might take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 5.7G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 4.5 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 50%|█████     | 16/32 [00:02<00:01,  9.36it/s]
We will save to Disk and not RAM now.
100%|██████████| 32/32 [01:57<00:00,  3.69s/it]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving model/pytorch_model-00001-of-00004.bin...
Unsloth: Saving model/pytorch_model-00002-of-00004.bin...
Unsloth: Saving model/pytorch_model-00003-of-00004.bin...
Unsloth: Saving model/pytorch_model-00004-of-00004.bin...
Done.


Unsloth: Converting llama model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at model into q8_0 GGUF format.
The output location will be /content/model/unsloth.Q8_0.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: model
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'pytorch_model.bin.index.json'
INFO:hf-to-gguf:gguf: loading model part 'pytorch_model-00001-of-00004.bin'
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> Q8_0, shape = {4

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Conversion completed! Output location: /content/model/unsloth.Q8_0.gguf
Unsloth: Saved Ollama Modelfile to model/Modelfile


#### Загрузка модели на HF

In [ ]:
if True: model.push_to_hub_gguf("katya1836/model", tokenizer, token = "hf_ВАШ_ТОКЕН_HUGGING_FACE")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 5.02 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 32/32 [01:29<00:00,  2.81s/it]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving katya1836/model/pytorch_model-00001-of-00004.bin...
Unsloth: Saving katya1836/model/pytorch_model-00002-of-00004.bin...
Unsloth: Saving katya1836/model/pytorch_model-00003-of-00004.bin...
Unsloth: Saving katya1836/model/pytorch_model-00004-of-00004.bin...
Done.
==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: [1] Converting model at katya1836/model into q8_0 GGUF format.
The output location will be /content/katya1836/model/unsloth.Q8_0.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: model
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:

  0%|          | 0/1 [00:00<?, ?it/s]

unsloth.Q8_0.gguf:   0%|          | 0.00/8.54G [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Saved GGUF to https://huggingface.co/katya1836/model


No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Saved Ollama Modelfile to https://huggingface.co/katya1836/model


Установка и запуск Ollama с помощью Python

In [ ]:
!sudo apt-get install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh
from IPython.display import clear_output

import os
import threading
import subprocess
import requests
import json

def ollama():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'
    subprocess.Popen(["ollama", "serve"])

ollama_thread = threading.Thread(target=ollama)
ollama_thread.start()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils
0 upgraded, 3 newly installed, 0 to remove and 21 not upgraded.
Need to get 343 kB of archives.
After this operation, 1,581 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 pci.ids all 0.0~2022.01.22-1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Fetched 343 kB in 1s (306 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: unable to initializ

In [ ]:
!ollama pull hf.co/katya1836/model

Error: could not connect to ollama app, is it running?


In [ ]:
prompt="""
### Question:
I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?

### Corresponding answer:
"""

response = requests.post(
    'http://localhost:11434/api/generate',
    json={
        'model': "hf.co/katya1836/model",
        "options": { "temperature": 0, "num_predict": 150},
        "prompt": prompt,
        "stream": False
    }
)

# Декодируем содержимое ответа
result = response.content.decode('utf-8')
result_json = json.loads(result)
# Извлекаем текст из ключа 'response'
generated_text = result_json.get('response')
print(generated_text)

None


#### Загрузка модели на Ollama

Теперь мы можем выполнить инференс с использованием этой модели через Ollama!

In [ ]:
!sudo apt-get install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh
from IPython.display import clear_output

import os
import threading
import subprocess
import requests
import json

def ollama():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'
    subprocess.Popen(["ollama", "serve"])

ollama_thread = threading.Thread(target=ollama)
ollama_thread.start()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
0 upgraded, 0 newly installed, 0 to remove and 21 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
############################################################################################# 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Ollama требует файл модели (Modelfile), который указывает формат подсказок (prompts) для модели. Давайте выведем автоматически сгенерированный файл модели от Unsloth

In [ ]:
print(tokenizer._ollama_modelfile)

FROM {__FILE_LOCATION__}

TEMPLATE """Below are some conversations. Write responses that appropriately complete each request.{{ if .Prompt }}

### Question:
{{ .Prompt }}{{ end }}

### Corresponding answer:
{{ .Response }}<|end_of_text|>"""

PARAMETER stop "<|start_header_id|>"
PARAMETER stop "<|end_header_id|>"
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end_of_text|>"
PARAMETER stop "<|reserved_special_token_"
PARAMETER temperature 1.5
PARAMETER min_p 0.1


Теперь мы создадим модель Ollama под названием unsloth_model, используя Modelfile, который был автоматически сгенерирован!

In [ ]:
!ollama create unsloth_model -f ./model/Modelfile

In [ ]:
!ollama run unsloth_model

In [ ]:
prompt="""
### Question:
I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?

### Corresponding answer:
"""

response = requests.post(
    'http://localhost:11434/api/generate',
    json={
        'model': "unsloth_model",
        "options": { "temperature": 0, "num_predict": 150},
        "prompt": prompt,
        "stream": False
    }
)

# Декодируем содержимое ответа
result = response.content.decode('utf-8')
result_json = json.loads(result)
# Извлекаем текст из ключа 'response'
generated_text = result_json.get('response')
print(generated_text)

#### Имитация работы через командную строку

In [ ]:
from IPython.display import JSON
from google.colab import output
from subprocess import getoutput
import os

def shell(command):
  if command.startswith('cd'):
    path = command.strip().split(maxsplit=1)[1]
    os.chdir(path)
    return JSON([''])
  return JSON([getoutput(command)])
output.register_callback('shell', shell)

In [ ]:
#@title Colab Shell
%%html
<div id=term_demo></div>
<script src="https://code.jquery.com/jquery-latest.js"></script>
<script src="https://cdn.jsdelivr.net/npm/jquery.terminal/js/jquery.terminal.min.js"></script>
<link href="https://cdn.jsdelivr.net/npm/jquery.terminal/css/jquery.terminal.min.css" rel="stylesheet"/>
<script>
  $('#term_demo').terminal(async function(command) {
      if (command !== '') {
          try {
              let res = await google.colab.kernel.invokeFunction('shell', [command])
              let out = res.data['application/json'][0]
              this.echo(new String(out))
          } catch(e) {
              this.error(new String(e));
          }
      } else {
          this.echo('');
      }
  }, {
      greetings: 'Welcome to Colab Shell',
      name: 'colab_demo',
      height: 250,
      prompt: 'colab > '
  });